# 17 — Corporate Finance Readiness Advisor

> A company brief goes in; the agent scores it across five dimensions — governance, financials, market position, legal, and investor narrative — and returns a prioritised action plan with a realistic time-to-ready estimate.

*Run all cells. Swap the input in the final code cell with your own data.*

In [ ]:
%pip install -q langchain-openai langchain-core pydantic python-dotenv
import os
os.environ['OPENAI_API_KEY'] = 'sk-...'  # replace

In [ ]:
from typing import List, Literal, Optional

from pydantic import BaseModel, Field
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_openai import ChatOpenAI


# --- Data shapes ---

class DimensionAssessment(BaseModel):
    dimension: Literal["governance", "financials", "market_position", "legal", "narrative"]
    score: int = Field(description="0-10 readiness score for this dimension")
    gate: Literal["pass", "conditional", "fail"] = Field(
        description=(
            "pass=ready now, conditional=fixable within 6 months, "
            "fail=structural blocker requiring major remediation"
        )
    )
    strengths: List[str] = Field(description="Positive factors in this dimension")
    blockers: List[str] = Field(
        description="Specific issues that must be resolved before transaction"
    )
    remediation: List[str] = Field(
        description="Concrete steps to address each blocker"
    )


class ReadinessReport(BaseModel):
    """Capital markets transaction readiness assessment across five dimensions."""

    company: Optional[str] = None
    transaction_type: Literal["ipo", "series_a", "series_b", "growth_equity", "pe_buyout"]
    overall_status: Literal["ready", "ready_with_conditions", "not_ready"] = Field(
        description=(
            "ready=all gates pass, ready_with_conditions=one or more conditional but no fail, "
            "not_ready=at least one fail gate"
        )
    )
    executive_summary: str = Field(
        description="3-4 sentences: overall readiness verdict and the single biggest blocker"
    )
    dimensions: List[DimensionAssessment] = Field(
        description="All five dimensions assessed, in order: governance, financials, "
        "market_position, legal, narrative"
    )
    critical_path: List[str] = Field(
        description="Ordered actions the company must take to reach 'ready' status"
    )
    estimated_time_to_ready: str = Field(
        description="Realistic estimate, e.g. '4-6 months', 'ready now', '12+ months'"
    )
    key_value_drivers: List[str] = Field(
        description="What makes this company attractive to investors despite the blockers"
    )


# --- Advisor logic ---

ADVISOR_SYSTEM = SystemMessage(
    """You are a corporate finance advisor assessing a company's readiness for a
capital markets transaction (IPO, Series A/B, growth equity, or PE buyout).

Evaluate the company across FIVE dimensions in this exact order:
  1. governance      -- board composition, audit committee, financial controls
  2. financials      -- quality of earnings, audit status, management accounts
  3. market_position -- TAM, competitive moat, customer concentration, NRR
  4. legal           -- IP ownership, contracts, regulatory status, litigation
  5. narrative       -- investor story, comparable transactions, team credibility

For each dimension:
  - score 0-10 (10 = fully ready, 0 = not started)
  - assign a GATE:
      "pass"        = ready now, no blockers
      "conditional" = fixable within 6 months with defined steps
      "fail"        = structural blocker requiring major remediation (>6 months)
  - list specific strengths
  - list specific blockers (name the exact issue, e.g. "IP assignment missing for 2 contractors")
  - list concrete remediation steps for each blocker

OVERALL STATUS RULE (apply exactly):
  - overall_status = "ready"                if ALL five gates are "pass"
  - overall_status = "ready_with_conditions" if one or more gates are "conditional" and NONE are "fail"
  - overall_status = "not_ready"            if ANY gate is "fail"

Provide a critical_path: ordered list of the most important actions to reach "ready" status."""
)


def run(company_brief: str) -> ReadinessReport:
    """
    Multi-dimension transaction readiness advisor: evaluates governance, financials,
    market_position, legal, and narrative with a go/no-go gate per dimension.

    Returns:
        ReadinessReport with dimensional scores, gates, and prioritized critical path
    """
    llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
    advisor = ADVISOR_SYSTEM | llm.with_structured_output(ReadinessReport)
    return advisor.invoke(
        HumanMessage(content="Company brief:\n\n" + company_brief)
    )

## The scenario

We're reviewing a health-tech SaaS company preparing for a PE buyout. The business has strong recurring revenue and a defensible niche in NHS-adjacent workflow automation, but the founding team has never run a formal sale process and the company's corporate housekeeping reflects that — no independent directors, no recent audit, and IP that was partly developed by a third-party agency without a formal assignment. The agent will score each dimension, flag every blocker that would derail a buyer's due diligence, and tell us the exact sequence of remediation steps before engaging a buyside mandate.

In [ ]:
COMPANY_BRIEF = """
COMPANY: Meridian Health Systems Ltd

Business: Workflow automation SaaS for NHS community health trusts and private GP networks.
Founded: 2017  |  HQ: Leeds, UK  |  Employees: 52

FINANCIAL PROFILE:
  ARR: GBP 6.4m  (+41% YoY)
  Gross margin: 71%
  Net revenue retention: 109%
  EBITDA: GBP 480k (7.5% margin, first profitable year)
  Last audited accounts: FY2021 (FY2022, FY2023 management accounts only)

TRANSACTION TARGET:
  PE buyout: seeking GBP 18-22m enterprise value
  Founders seeking partial liquidity; management to roll equity

GOVERNANCE:
  Directors: CEO (founder), COO (co-founder), FD (part-time, 3 days/week)
  No independent non-executive directors
  No audit committee; board meets informally every 6-8 weeks
  No formal risk register or internal controls framework

LEGAL / IP:
  Core scheduling engine built 2017-2019 by an external dev agency (Leeds-based).
  Agency contract does not include an IP assignment clause.
  All post-2019 development done in-house with signed employment contracts.
  Customer contracts: bespoke NHS framework agreements; 3-year terms typical.
  One outstanding contract dispute with a former reseller (GBP 85k claim, pre-litigation).

MARKET:
  148 NHS and private GP customers (largest = 18% of ARR)
  Addressable market: UK primary care workflow software (GBP 1.1bn)
  Main competitors: 2 larger legacy vendors; Meridian differentiates on ease of deployment
  NRR 109%; annual churn below 4%

NARRATIVE:
  No formal CIM or information memorandum prepared.
  FD has no prior M&A transaction experience.
  CEO has relationships with 2 sector-specialist PE firms from a prior approach in 2022.
  No management presentation deck; historical board packs are the only materials available.
"""

report = run(COMPANY_BRIEF)

GATE_LABEL = {"pass": "PASS", "conditional": "COND", "fail": "FAIL"}

company = report.company or "Meridian Health Systems"
print("=" * 65)
print(f"READINESS REPORT  |  {company}  |  {report.transaction_type.upper()}")
print("=" * 65)
print(f"\nOverall status: {report.overall_status.upper()}")
print(f"\n{report.executive_summary}")

print("\nDIMENSION SCORECARD:")
for d in report.dimensions:
    label = GATE_LABEL[d.gate]
    print(f"\n  [{label}] {d.dimension.upper()}  --  {d.score}/10")
    if d.strengths:
        print(f"  Strengths: {'; '.join(d.strengths)}")
    if d.blockers:
        print(f"  Blockers:  {'; '.join(d.blockers)}")
    if d.remediation:
        print("  Remediation:")
        for step in d.remediation:
            print(f"    - {step}")

print(f"\nCRITICAL PATH ({len(report.critical_path)} actions):")
for i, action in enumerate(report.critical_path, 1):
    print(f"  {i}. {action}")

print(f"\nTime to ready: {report.estimated_time_to_ready}")

if report.key_value_drivers:
    print("\nKEY VALUE DRIVERS:")
    for v in report.key_value_drivers:
        print(f"  + {v}")

## Use your own data

Replace the `COMPANY_BRIEF` string above with:
- Your company overview: sector, headcount, founding year, HQ
- Financials: ARR or revenue, margin, growth rate, audit status, runway
- Governance: directors, board structure, committee status
- Legal: IP ownership history, key contracts, any open disputes
- Market: customer count, top-customer concentration, competitive position
- Narrative: existing investor materials, management M&A experience

The agent returns a score and gate verdict for each of the five dimensions, a ranked list of the actions that must be completed before a deal can close, and a realistic time-to-ready estimate.